In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import os

# Hardcoded Activity mappings directly from your provided HTML block 
# to ensure stability and reduce DOM stale errors during rapid resets.
ACTIVITIES = {
    "6": "1(d) Thermal Power Plants",
    "10": "3(a) Metallurgical Industries (ferrous and non ferrous)", 
    "11": "3(b) Cement plants", 
    "12": "4(a) Petroleum refining industry", 
    "13": "4(b) Coke oven plants",
    "19": "5(a) Chemical fertilizers", 
    "21": "5(c) Petro-chemical complexes (industries based on processing of petroleum fractions",
    "25": "5(g) Distilleries",
    "27": "5(i) Pulp & Paper Industry", 
    "28": "5(j) Sugar Industry",
    "75": "5(ga) Grain based distilleries", 
    "79": "2(c) Pellet Plant"
}

driver = webdriver.Chrome()
driver.get("https://parivesh.nic.in/newupgrade/#/trackYourProposal/")
wait = WebDriverWait(driver, 20)

advance_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Show Advance Search')]")))
advance_btn.click()

# -----------------------------
# Select Major Clearance Type
# -----------------------------
major_clearance = wait.until(EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='majorClearanceType']")))
Select(major_clearance).select_by_value("1")

dropdown_element = driver.find_element(By.CSS_SELECTOR, "select[formcontrolname='issueAuthority']")
select = Select(dropdown_element)
select.select_by_value("SEIAA")

# Initialize data accumulation storage
all_table_data = []
headers = []  

# -----------------------------
# 1. Get total number of states
# -----------------------------
state_dropdown = wait.until(EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='state']")))
total_states = len(Select(state_dropdown).options)
print(f"Total State options found: {total_states}")

# -----------------------------
# 2. Nested State & Activity Loops
# -----------------------------
for state_idx in range(1, total_states):  # Outer loop: States
    state_dropdown = wait.until(EC.visibility_of_element_located((By.XPATH, "//select[@formcontrolname='state']")))
    state_select = Select(state_dropdown)
    state_value = state_select.options[state_idx].get_attribute("value")
    state_select.select_by_index(state_idx)
    
    print(f"\n🌍 Processing State: Index {state_idx} (Value: {state_value})...")
    
    for act_value, act_desc in ACTIVITIES.items():  # Inner loop: Activities
        print(f"  └── ⚙️ Searching Activity ID: {act_value} ({act_desc[:30]}...)")
        
        try:
            # Re-locate components per iteration to prevent stale exceptions
            activity_dropdown = wait.until(EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='activityId']")))
            activity_select = Select(activity_dropdown)
            activity_select.select_by_value(act_value)
        except Exception as e:
            print(f"  ⚠️ Could not select Activity {act_value}: {e}. Skipping activity...")
            continue

        # Click Search Button Safely
        search_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(.,'Search')]")))
        
        existing_tables = driver.find_elements(By.ID, "excel-table")
        old_table = existing_tables[0] if existing_tables else None

        driver.execute_script("arguments[0].click();", search_button)
        
        if old_table:
            try:
                wait.until(EC.staleness_of(old_table))
            except Exception:
                time.sleep(1) 

        # Handle Missing Table if No Results Exist
        try:
            wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
        except TimeoutException:
            print(f"  ℹ️ No results found for State: {state_value} + Activity: {act_value}. Proceeding...")
            continue

        # -----------------------------
        # LIVE ROW SCRAPING LOGIC 
        # -----------------------------
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        table = soup.find("table", {"id": "excel-table"})
        
        if table:
            tbody = table.find("tbody")
            if tbody:
                rows_bs = tbody.find_all("tr")
                tbody_text = tbody.get_text(strip=True).lower()
                
                if "no record" in tbody_text or "no data" in tbody_text or not rows_bs:
                    continue

                if not headers:
                    thead = table.find("thead")
                    if thead:
                        for th in thead.find_all("th"):
                            headers.append(th.get_text(strip=True))

                total_rows = len(rows_bs)
                print(f"  📊 Found {total_rows} records matching criteria.")

                # Loop through table rows sequentially
                for row_idx in range(1, total_rows + 1):
                    try:
                        cols_elements = driver.find_elements(By.XPATH, f"//table[@id='excel-table']/tbody/tr[{row_idx}]/td")
                        row_data = [col.text.strip() for col in cols_elements]
                        
                        if not row_data:
                            continue
                        
                        # Append metadata markers to the data structure
                        row_data.append(state_value)        
                        row_data.append(act_desc)        # Adds the plain text Activity description
                        all_table_data.append(row_data)
                        
                    except Exception as e:
                        print(f"   ❌ Error processing row index {row_idx}: {e}")
                        continue

        time.sleep(0.5)

# -----------------------------
# 3. Post-Loop Data Compilation
# -----------------------------
if all_table_data:
    expected_header_count = len(all_table_data[0])
    
    # Pad headers sequentially to match the appended data columns
    if len(headers) < expected_header_count:
        headers.append("State_Value")
    if len(headers) < expected_header_count:
        headers.append("Activity Description")
        
    df = pd.DataFrame(all_table_data, columns=headers)
    print("\n--- Final Extracted Dataset Preview ---")
    print(df.head()) 

    file_path = "parivesh_data.csv"
    file_exists = os.path.exists(file_path)

    df.to_csv(
        file_path, 
        mode='a', 
        index=False, 
        header=not file_exists  
    )
    print(f"\nScraping complete! Combined runs saved to {file_path}")
else:
    print("\n❌ Automation complete. Zero data entries found across the state/activity matrix.")

driver.quit()

Total State options found: 37

🌍 Processing State: Index 1 (Value: 35)...
  └── ⚙️ Searching Activity ID: 6 (1(d) Thermal Power Plants...)
  ℹ️ No results found for State: 35 + Activity: 6. Proceeding...
  └── ⚙️ Searching Activity ID: 10 (3(a) Metallurgical Industries ...)
  ℹ️ No results found for State: 35 + Activity: 10. Proceeding...
  └── ⚙️ Searching Activity ID: 11 (3(b) Cement plants...)
  ℹ️ No results found for State: 35 + Activity: 11. Proceeding...
  └── ⚙️ Searching Activity ID: 12 (4(a) Petroleum refining indust...)
  ℹ️ No results found for State: 35 + Activity: 12. Proceeding...
  └── ⚙️ Searching Activity ID: 13 (4(b) Coke oven plants...)
  ℹ️ No results found for State: 35 + Activity: 13. Proceeding...
  └── ⚙️ Searching Activity ID: 19 (5(a) Chemical fertilizers...)
  ℹ️ No results found for State: 35 + Activity: 19. Proceeding...
  └── ⚙️ Searching Activity ID: 21 (5(c) Petro-chemical complexes ...)
  ℹ️ No results found for State: 35 + Activity: 21. Proceeding...
